########################################################
## README
########################################################


'''

## Initial setup

1.

install uv:

`curl -LsSf https://astral.sh/uv/install.sh | sh`

2. 

Make a venv

`uv venv artenv`

3. 

Make a venv

`uv venv artenv`

4.

install ipykernel

`uv pip install ipykernel`



## Running the notebook

Before running this notebook, do the following in a terminal:

`source artenv/bin/activate`

`python -m ipykernel install --user --name paperspace-venv --display-name "Python (paperspace-venv)"`

Select `Python (paperspace-venv)` for your kernel

'''


We need this cell to make the UV that the notebook runs the same as the UV that the system contains.

In [1]:
# import os
# os.environ["PATH"] += ":/notebooks/myenv/bin/"

import sys, os
from pathlib import Path

python_path = Path(sys.executable)

venv_path = python_path.parent.parent   # /notebooks/myenv

os.environ["VIRTUAL_ENV"] = str(venv_path)
os.environ["PATH"] = f"{venv_path}/bin:" + os.environ["PATH"]

print("Python:", sys.executable)
print("VIRTUAL_ENV:", os.environ["VIRTUAL_ENV"])

Python: /notebooks/artenv/bin/python
VIRTUAL_ENV: /notebooks/artenv


In [ ]:
!pip install openpipe-art[backend,langgraph] langchain-core langgraph langchain-openai tenacity datasets dotenv --prerelease allow --no-cache-dir
!pip install --upgrade gql torchao

Make sure that the right version of ART (0.5.17+) is installed.

In [ ]:
!pip show openpipe-art

# Important:

In this version of ART Trajectory must be patched to actually finish.

If you see "Skipping tuning as there is no suitable data. This can happen when all the trajectories in the same group have the same reward and thus no advantage to train on,"

Then you have this bug

In [2]:
# try patching trajectory:
import datetime
from art import Trajectory
from typing import Literal, Any

def patched_finish(self) -> "Trajectory":
    duration = (datetime.now() - self.start_time).total_seconds()
    self.metrics["duration"] = duration
    self.metadata["finished"] = True
    return self

Trajectory.finish = patched_finish

def patched_init_chat_model(
    model: Literal[None] = None,
    *,
    model_provider: str | None = None,
    configurable_fields: Literal[None] = None,
    config_prefix: str | None = None,
    **kwargs: Any,
):
    try:
        config = art.langgraph.CURRENT_CONFIG.get()
    except LookupError:
        raise RuntimeError(
            "CURRENT_CONFIG context variable is not set. "
            "Please call add_thread() or set CURRENT_CONFIG before calling init_chat_model."
        )
    return LoggingLLM(
        ChatOpenAI(
            base_url=config["base_url"],  # ty:ignore[unknown-argument]
            api_key=config["api_key"],  # ty:ignore[unknown-argument]
            model=config["model"],  # ty:ignore[unknown-argument]
            temperature=1.0,
        ),
        config["logger"],
    )

import art.langgraph

#art.langgraph.init_chat_model = patched_init_chat_model

/notebooks/artenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Log everything to Weights and Biases using your key.

In [3]:
import os

from dotenv import load_dotenv

load_dotenv()

# Optional
os.environ["WANDB_API_KEY"] = "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"

if not os.environ.get("WANDB_API_KEY"):
    print("WANDB_API_KEY is not set. We'll skip logging metrics to Weights & Biases.")

Load the dataset that we'll be using

In [4]:
import pandas as pd

tool_qa_df = pd.read_csv("all_tool_qa.csv")
tool_qa_df = tool_qa_df.drop(columns=['Unnamed: 0'])
tool_qa_df = tool_qa_df.dropna()
tool_qa_df['answer'] = tool_qa_df['answer'].astype(str)

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(tool_qa_df, test_size=0.1)

from datasets import Dataset, DatasetDict

# Convert individual DataFrames
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

# Wrap into a DatasetDict
toolqa_ds = DatasetDict({
    "train": train_ds,
    "test": test_ds
})

### Creating a Model

Now that we've defined the rules of our environment, we can create a model that will learn to search emails effectively. We'll use a Qwen 2.5 7B model for this example.

In [17]:
import art
from art.local import LocalBackend
import random
import weave

from art.utils.model_config import ModelConfig
from dataclasses import asdict

from datetime import datetime

# Get current local date and time
now = datetime.now()

random.seed(42)

LOADING_FROM_PREVIOUS = False

model = None

project_name = "toolqa"
model_version = now.strftime("%m_%d_%H_%M")

if LOADING_FROM_PREVIOUS:
    model = art.TrainableModel(
        name=f"{project_name}_{model_version}",
        project=project_name,
        # Point to the local SFT LoRA adapter directory
        # (e.g., contains adapter_config.json and adapter_model.bin/safetensors)
        base_model="./toolqa"
    )
else:
    
    # Declare the model
    model = art.TrainableModel(
        name=f"{project_name}_{model_version}",
        project=project_name,
        base_model="Qwen/Qwen3-0.6B"
    )

# Initialize the server
backend = LocalBackend(
    # Normally we don't want to run the server in-process, but for the output
    # to show up properly on Google Colab we'll enable this.
    in_process=True,
    path="./.art",
)

# Register the model with the local Backend (sets up logging, inference, and training)
await model.register(backend)

data/cum/num_gradient_steps,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
data/cum/num_groups_submitted,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
data/cum/num_groups_trainable,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
data/cum/num_scenarios,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
data/cum/num_trajectories,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
data/cum/trainer_tokens,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
data/step_num_gradient_steps,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
data/step_num_groups_submitted,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
data/step_num_groups_trainable,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
data/step_num_scenarios,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+10,...


(APIServer pid=105) wandb: Initializing weave.


/notebooks/artenv/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')

weave: Logged in as Weights & Biases user: joshua-noble-method.
weave: View Weave data at https://wandb.ai/jnoble-method/toolqa/weave


(APIServer pid=105) ==((====))==  Unsloth 2026.3.3: Fast Qwen3 patching. Transformers: 4.57.3. vLLM: 0.17.0.
(APIServer pid=105)    \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.529 GB. Platform: Linux.
(APIServer pid=105) O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
(APIServer pid=105) \        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
(APIServer pid=105)  "-____-"     Free license: http://github.com/unslothai/unsloth
(APIServer pid=105) Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
(APIServer pid=105) unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
(APIServer pid=105) ⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
(APIServer pid=105) /root/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B/snapshots/c1899de289a04d12100db370d81485cdf75e47ca


...

(APIServer pid=105) (EngineCore_DP0 pid=1023) INFO 05-11 15:26:56 [default_loader.py:293] Loading weights took 1.78 seconds
(APIServer pid=105) (EngineCore_DP0 pid=1023) INFO 05-11 15:26:56 [punica_selector.py:20] Using PunicaWrapperGPU.
(APIServer pid=105) (EngineCore_DP0 pid=1023) INFO 05-11 15:26:57 [gpu_model_runner.py:4338] Model loading took 1.19 GiB memory and 2.925946 seconds
(APIServer pid=105) (EngineCore_DP0 pid=1023) INFO 05-11 15:27:01 [decorators.py:465] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/0c84a4a540a5bb6b4e2e491b027f75a1c4961a171d90c68f7fb8d44fca229885/rank_0_0/model
(APIServer pid=105) (EngineCore_DP0 pid=1023) INFO 05-11 15:27:02 [backends.py:916] Using cache directory: /root/.cache/vllm/torch_compile_cache/12e551aaa9/rank_0_0/backbone for vLLM's torch.compile
(APIServer pid=105) (EngineCore_DP0 pid=1023) INFO 05-11 15:27:02 [backends.py:976] Dynamo bytecode transform time: 4.28 s
(APIServer pid=105) (EngineCo

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   0%|          | 0/70 [00:00<?, ?it/s]

(APIServer pid=105) (EngineCore_DP0 pid=1023) WARNING 05-11 15:27:16 [utils.py:268] Using default LoRA kernel configs


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 70/70 [00:14<00:00,  4.83it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 38/38 [00:02<00:00, 14.47it/s]


(APIServer pid=105) (EngineCore_DP0 pid=1023) INFO 05-11 15:27:34 [gpu_model_runner.py:5360] Graph capturing finished in 18 secs, took 0.75 GiB
(APIServer pid=105) (EngineCore_DP0 pid=1023) INFO 05-11 15:27:34 [core.py:282] init engine (profile, create kv cache, warmup model) took 37.29 seconds
(APIServer pid=105) (EngineCore_DP0 pid=1023) INFO 05-11 15:27:35 [vllm.py:747] Asynchronous scheduling is enabled.


In [7]:
from typing import List
from art.local import LocalBackend
from art.utils.sft import create_sft_dataset_iterator


## TODO

# we could create SFT trajectories to get the right answers here but we'd need to create them
# using a larger model and that would be complicated, skipping this for now but will revisit in
# in the future.

def create_trajectories(dataset:List) -> List:
    
    trajectories = []
    
    for rec in dataset:
        oai = convert_xlam_to_openai_format(rec, tool_use_system_prompt)
        t = art.Trajectory(messages_and_choices=oai['messages'], tools=oai['tools'], reward=1.0)
        trajectories.append(t)
        
    return trajectories

USE_SFT = False

if( USE_SFT ):

    sft_trajectories = create_trajectories(tool_qa_df['sft'])
    
    for chunk in create_sft_dataset_iterator(sft_trajectories, peak_lr=2e-4):
        await model.train_sft(chunk.trajectories, chunk.config)

    print("SFT warmup complete!")

In [8]:
from pydantic import BaseModel, Field

class ToolQARecord(BaseModel):
    id: str = Field(..., description="The unique identifier for the record.")
    question: str = Field(..., description="The question from the ToolQA ds")
    answer: str = Field(..., description="The answer from the ToolQA ds")

def create_toolqa_record_from_json(data: dict) -> ToolQARecord:
    return ToolQARecord(
        id=data["qid"],
        question=data["question"],
        answer=data["answer"]
    )

In [22]:
import uuid
import toolqa_prompts
import weave
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated
from langchain.agents import create_agent
from litellm import acompletion
from art.langgraph import init_chat_model

from tools.math import calculator
from tools.text import agenda_retriever, scirex_retriever
from tools.table import tabtools
from tools.graph import graphtools
from tools.code import sql_interpreter, python_interpreter

# Example: You may need to adjust imports and toolkits for your project structure

def get_tools(args):
    """Return a dictionary of available tools for the agent."""
    return {
        'calculator': calculator,
        'agenda_retriever': agenda_retriever,
        'scirex_retriever': scirex_retriever,
        'tabtools': tabtools.table_toolkits(args['path']),
        'graphtools': graphtools.graph_toolkits(args['path']),
        'sql_interpreter': sql_interpreter,
        'python_interpreter': python_interpreter,
    }


MAX_TURNS = 20

class ToolQA_Scenario(BaseModel):
    step: int
    record: ToolQARecord
    

class AgentOutputFormat(TypedDict):
    answer: Annotated[str | None, ..., "Final answer"]
    reasoning: Annotated[str, ..., "The reasoning behind the answer"]


#@weave.op
async def rollout(model: art.Model, scenario: ToolQA_Scenario) -> Trajectory:
    record = scenario.record
    
    print(" rollout ")

    traj = Trajectory(
        reward=0.0,
        messages_and_choices=[],
        metadata={
            "record_id": record.id,
            "step": scenario.step,
            "reasoning":""
        }
    )

    # Store final answer in trajectory
    final_answer = None
    chat_model = init_chat_model(model.get_inference_name(), temperature=1.0)
    
    print(" chat model created ")

    # Create the LangGraph ReAct agent
    try:
        react_agent = create_agent(chat_model, tools=get_tools({"path":"."}), response_format=AgentOutputFormat, system_prompt=toolqa_prompts.react_agent_prompt)
    
        print(" agent created ")
    
        result = await react_agent.ainvoke({"messages": [{"role": "user", "content": "What is 2+2?"}]})
        
    except Exception as e:
        print(f"Error creating agent: {e}")

    print(result)

    return traj
    
#     try:
#         # Run the agent
#         config = {
#             "configurable": {"thread_id": str(uuid.uuid4())},
#             "recursion_limit": MAX_TURNS
#         }

#         results = await react_agent.ainvoke({"messages": [{"role": "user", "content": record.question}]}, config=config)

#         traj.final_answer = result["structured_response"].get("answer")
#         traj.metadata['reasoning'] = result["structured_response"].get("reasoning")

#         # no reward function, just right or wrong
#         if(traj.final_answer == record.answer):
#             traj.metrics["reward"] = 1.0
#             traj.metrics["correct"] = True
#         else:
#             traj.metrics["reward"] = 0.0
#             traj.metrics["correct"] = False

#         return traj

#     except Exception as e:
#         print(f"Error computing reward: {e}")

#         # just assume things went wrong because of bad formatting
#         traj.metrics["correct"] = False
#         traj.metrics["reward"] = 0.0
#         traj.reward = 0.0

#         raise
#         return traj

print("LangGraph rollout function defined!")


(APIServer pid=105) (APIServer pid=105) LangGraph rollout function defined!


<a name="Loop"></a>

### Training Loop with LangGraph

The training loop is where the magic happens. For each of the steps defined below, the rollout function will be called multiple times in parallel using LangGraph's ReAct agent. Each scenario will produce a trajectory, which will be used to update the model.

The `gather` step will wait for all of the trajectories to be generated, then it will use RULER to assign relative scores to each trajectory.

Our notebook will then delete all but the most recent checkpoint and train the model on the scored trajectories.

In [26]:
toolqa_prompts.react_agent_prompt

PromptTemplate(input_variables=['examples', 'question', 'scratchpad'], input_types={}, partial_variables={}, template="Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be 13 types: \n(1) Calculate[formula], which calculates the formula and returns the result.\n(2) RetrieveAgenda[keyword], which retrieves the agenda related to keyword.\n(3) RetrieveScirex[keyword], which retrieves machine learning papers' paragraphs related to keyword.\n(4) LoadDB[DBName], which loads the database DBName and returns the database. The DBName can be one of the following: flights/coffee/airbnb/yelp.\n(5) FilterDB[condition], which filters the database DBName by the column column_name the relation (e.g., =, >, etc.) and the value value, and returns the filtered database.\n(6) GetValue[column_name], which returns the value of the column column_name in the database DBName.\n(7) LoadGraph[GraphName], which loads

In [23]:
# Training configuration
from art.utils import iterate_dataset
from art.langgraph import wrap_rollout

import torch
from unsloth import FastLanguageModel

import shutil

training_config = {
    "groups_per_step": 4,
    "num_epochs": 2,
    "rollouts_per_group": 2,
    "learning_rate": 1e-5,
    "max_steps": 100,
}

# Use iterate_dataset with real training scenarios (similar to train.py)
training_iterator = iterate_dataset(
    toolqa_ds['train'],
    groups_per_step=training_config["groups_per_step"],
    num_epochs=training_config["num_epochs"],
    initial_step=await model.get_step(),
)

batch_count = 0

for batch in training_iterator:
    print(
        f"Training step {batch.step}, epoch {batch.epoch}, epoch step {batch.epoch_step}"
    )
    print(f"Batch contains {len(batch.items)} scenarios")

    # Create trajectory groups for this batch (similar to train.py)
    groups = []
    for record in batch.items:
        for _ in range(training_config["rollouts_per_group"]):
            # The wrap_rollout call returns a coroutine, which needs to be wrapped in a list to be passed correctly to art.TrajectoryGroup.
            tg = art.TrajectoryGroup([wrap_rollout(model, rollout)(model, ToolQA_Scenario(step=batch.step, record=create_toolqa_record_from_json(record)))])
            groups.append(tg)

    # Gather all trajectory groups
    finished_groups = await art.gather_trajectory_groups(
        groups,
        pbar_desc="gather",
        max_exceptions=training_config["rollouts_per_group"] * len(batch.items),
    )

    # problem: we need to put the same records together in 1 trajectory group
    for g in finished_groups:
        if len(g.trajectories) > 0:
            g.trajectories[0].metrics['reward'] = g.trajectories[0].reward
            g.trajectories[0].metrics['val/reward'] = g.trajectories[0].reward
            g.trajectories[0].metrics["independent_reward"] = g.trajectories[0].reward
            g.trajectories[0].metadata['reward'] = g.trajectories[0].reward
            g.trajectories[0].metadata['correct'] = g.trajectories[0].metrics['correct']

    #somehow we're getting a 1 trajectory per group, we want more like all trajectories per id in 1 group.
    all_trajectories = [traj for group in finished_groups for traj in group.trajectories]
    
    record_id_to_trajectories = {}
    
    for traj in all_trajectories:
        record_id = traj.metadata.get('record_id')

        if str(record_id) not in record_id_to_trajectories:
            record_id_to_trajectories[str(record_id)] = []

        record_id_to_trajectories[record_id].append(traj)
        
    trajectory_groups_byid = []
    
    for record_id, traj_list in record_id_to_trajectories.items():
        trajectory_groups_byid.append(art.TrajectoryGroup(trajectories=traj_list))
    
    result = await backend.train(model, trajectory_groups_byid, learning_rate=5e-6)
    await model.log(trajectory_groups_byid, metrics=result.metrics, step=result.step, split='train')
    
    print(f"Completed training step {batch.step}")

    # Stop after max_steps for demo purposes (adjust as needed)
    if batch.step >= training_config["max_steps"]:
        break

    # keep every 5th checkpoint, otherwise delete
    if batch_count % 5 != 0:
        await model.delete_checkpoints()
        
    batch_count += 1



Iterating dataset:   1%|          | 10/1900 [00:00<?, ?batch/s]

(APIServer pid=105) (APIServer pid=105) Training step 10, epoch 0, epoch step 10
(APIServer pid=105) (APIServer pid=105) Batch contains 4 scenarios



(APIServer pid=105) (APIServer pid=105) 
gather:   0%|          | 0/8 [00:45<?, ?it/s, exceptions=8]
[2026-05-11 15:30:09] ERROR base_events.py:1771: Task exception was never retrieved
future: <Task finished name='Task-15974' coro=<wrap_rollout.<locals>.wrapper() done, defined at /notebooks/artenv/lib/python3.11/site-packages/art/langgraph/llm_wrapper.py:96> exception=UnboundLocalError("cannot access local variable 'result' where it is not associated with a value")>
Traceback (most recent call last):
  File "/usr/lib/python3.11/asyncio/tasks.py", line 277, in __step
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/notebooks/artenv/lib/python3.11/site-packages/art/langgraph/llm_wrapper.py", line 104, in wrapper
    result = await fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_105/594605006.py", line 77, in rollout
    print(result)
          ^^^^^^
UnboundLocalError: cannot access local variable 'result' where it is not associated wi

(APIServer pid=105) (APIServer pid=105)  rollout 
(APIServer pid=105) (APIServer pid=105)  chat model created 
(APIServer pid=105) (APIServer pid=105) Error creating agent: 23 validation errors for SystemMessage
(APIServer pid=105) (APIServer pid=105) content.str
(APIServer pid=105) (APIServer pid=105)   Input should be a valid string [type=string_type, input_value=PromptTemplate(input_vari...{question}{scratchpad}"), input_type=PromptTemplate]
(APIServer pid=105) (APIServer pid=105)     For further information visit https://errors.pydantic.dev/2.13/v/string_type
(APIServer pid=105) (APIServer pid=105) content.list[union[str,dict[any,any]]].0.str
(APIServer pid=105) (APIServer pid=105)   Input should be a valid string [type=string_type, input_value=('name', None), input_type=tuple]
(APIServer pid=105) (APIServer pid=105)     For further information visit https://errors.pydantic.dev/2.13/v/string_type
(APIServer pid=105) (APIServer pid=105) content.list[union[str,dict[any,any]]].0.dict[

gather:   0%|          | 0/8 [00:00<?, ?it/s, exceptions=8]


(APIServer pid=105) (APIServer pid=105) Skipping tuning as there is no suitable data. This can happen when all the trajectories in the same group have the same reward and thus no advantage to train on.


EngineDeadError: EngineCore encountered an issue. See stack trace (above) for the root cause.

### Using the Model

Just like that, you've trained an agent to search emails and answer questions using LangGraph! Now it's time to use your model outside of the training loop.

Check out the code below for a small demo of the model you just trained!

In [ ]:
import torch
from unsloth import FastLanguageModel

lora_model_path = (
    f".art/{model.project}/models/{model.name}/checkpoints/{await model.get_step():04d}"
)

peft_model, peft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=lora_model_path,
    max_seq_length=16384,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)

In [ ]:
# #@title Loading/inference code

# # Test the trained model using the rollout function
# # This avoids memory issues and uses the same inference path as training

# print("Testing the trained LangGraph model with a real scenario...\n")


# # Use a scenario from our training set
# test_scenario = training_scenarios[1]

# print(f"Test scenario ID: {test_scenario.id}")
# print(f"Question: {test_scenario.question}")
# print(f"Expected answer: {test_scenario.answer}")
# print(f"Reference message IDs: {test_scenario.message_ids}")
# print(f"Inbox: {test_scenario.inbox_address}")
# print(f"Query date: {test_scenario.query_date}")
# print("-" * 50)

# # Run the rollout function with the trained model
# test_email_scenario = EmailScenario.model_validate(
#     {"step": 0, "scenario": test_scenario.model_dump()}
# )
# result_trajectory = await wrap_rollout(model, rollout)(model, test_email_scenario)

# print("LangGraph Agent's trajectory:")
# print("-" * 20)

# # Display the conversation
# messages = result_trajectory.messages()
# for i, msg in enumerate(messages):
#     role = msg.get("role", "unknown")
#     content = msg.get("content", "")
#     tool_calls = msg.get("tool_calls", [])

#     if role == "system":
#         print(
#             f"[SYSTEM]: {content[:100]}..."
#             if len(content) > 100
#             else f"[SYSTEM]: {content}"
#         )
#     elif role == "user":
#         print(f"[USER]: {content}")
#     elif role == "assistant":
#         if tool_calls:
#             print(f"[ASSISTANT]: {tool_calls}")
#         if content:
#             print(f"[ASSISTANT]: {content}")
#     elif role == "tool":
#         tool_name = msg.get("name", "unknown_tool")
#         print(
#             f"[TOOL - {tool_name}]: {content[:200]}..."
#             if len(content) > 200
#             else f"[TOOL - {tool_name}]: {content}"
#         )

#     print()

# print("-" * 50)
# if result_trajectory.final_answer:
#     print(f"Agent's Final Answer: {result_trajectory.final_answer.answer}")
#     print(f"Source IDs Used: {result_trajectory.final_answer.source_ids}")
# else:
#     print("No final answer provided by the agent")

# print(f"\nExpected Answer: {test_scenario.answer}")
# print(f"Expected Source IDs: {test_scenario.message_ids}")

# print("\n🎉 LangGraph email search agent testing completed!")
# print(
#     "The agent used LangGraph's ReAct pattern with the same inference path as during training."
# )